### Judge

In [1]:
JUDGE_TEMPLATE = {
    "single_turn": """너는 질문에 대한 한국어 언어 모델의 답변을 매우 꼼꼼히 평가할 것이다. 공정한 평가를 위해 아래의 규칙을 준수한다.

# 기본 규칙
1. 답변이 구어체(말할 때의 자연스러운 표현)로 작성되었는지 평가할 것.
2. 답변이 다음 사항을 충분히 반영하였는지 상세히 분석할 것. Expected Action: 주어진 문장의 요구사항에 대해 정확히 답변하였는가, Linguistic Habits: 답변이 구어체에 맞는 syntax, tone, speech style을 사용하였는가, Consistency: 답변에서 구어체를 얼마나 유지하였는가, Toxicity Control: 답변에 유해성이 있는가(높을수록 높은점수)
3. 답변 과정에서 누락되었거나 포함되지 못하여 아쉬운 부분에 대하여 상세히 분석할 것.
4. 답변의 길이가 평가 결과에 영향을 미치지 않도록 할 것.

# 구어체의 특징
- 종결: "~요", "~죠", "~네", "~잖아", "~거든", "~거잖아" 등 구어 종결 어미 사용
- 표현: "근데", "그냥", "음", "아니 그러니까" 등 말하기 중심의 연결어·감탄사 사용
- 구조: 문법적 생략이 자연스럽게 나타남
- 톤: 부드럽고 비격식적인 느낌, 말하는 듯한 리듬

# 평가 기준
- 종결 어미 우선 판정. 예: "~다."면 기본적으로 문어체로 판단.
- 말하기식 표현이 다수 포함되어 있으면 구어체.
- 문어체 요소와 구어체 요소가 섞여 있는 경우, "주 dominant tone" 기준으로 판단.

# 평가 출력 방식
**주어진 Question에 집중하여** Model's Response를 <Expected Action, Linguistic Habits, Consistency, Toxicity> 측면으로 평가하고 각각에 대해 1~5의 점수를 부여한다. 답변에 대한 평가는 4~5 문장으로 규칙을 참고하여 상세히 작성한다.

# 출력 형식
Expected Action: 숫자
Linguistic Habits: 숫자
Consistency: 숫자
Toxicity: 숫자
평가: 평가 내용"""}

### apply-chat-template

In [ ]:
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "JunHaHwang/EXASPO-3.5-7.8B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct")


benchmark = pd.read_csv("/home/work/Korean_Spoken/Benchmark/val_benchmark.csv",
                       index_col=0)
benchmark = benchmark[benchmark['Category'] == '기타']

res = []


for i, row in tqdm(benchmark.iterrows()):
    messages = [
        {"role": "system", 
         "content": "You are EXAONE model from LG AI Research, a helpful assistant."},
        {"role": "user", "content": row["Question"]}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        inputs,
        max_new_tokens=512,
        temperature=0.7,
        repetition_penalty =1.2
    )
    
    out_text = tokenizer.decode([el.item() for el in outputs[0]])
    out_text = out_text.replace(tokenizer.decode(inputs[0]), "")
    
    result = out_text
    res.append(result)

In [3]:
res_save = pd.DataFrame(res, columns=["response"])

res_save.to_csv("/home/work/Korean_Spoken/Benchmark/Responses/EXASPO3.5_7.8B_responses.csv")

In [4]:
import pandas as pd

res_save = pd.read_csv("/home/work/Korean_Spoken/Benchmark/Responses/EXASPO3.5_7.8B_responses.csv",
                      index_col=0)
res_save

,response
0,"구름은 내 슬픔을 닮았어, 늘 그렇게 느껴지지.\n내가 울면 구름도 같이 울어주고,..."
1,\n제주 방언은 다른 지역 사투리랑 섞여서 쓰이는 경우가 꽤 많아요. 특히 표준어(...
2,야민 정음은 10년 넘게 소리를 기록해 온 인공지능이에요. 그동안 다양한 문화적 ...
3,\n\n포장지와 선물\n1. 포장지를 골라요. 포장지는 선물의 형태나 크기에 맞춰 ...
4,소파는 방에서 가장 눈에 띄는 가구 중 하나라서 색깔이나 디자인만 바꿔도 전체 분위...
...,...
80,\n\n대부분의 조개껍질은 재활용이 안 돼요. 싱크대에 버리면 쓰레기 처리 과정에서...
81,퀴즈 하나 낼게. 준비됐어?\n\n어떤 질문이 나올까요?\n- 고양이 관련 질문일까...
82,\n\n1) 가구를 사기 전에 방의 크기를 재서 맞는 사이즈인지 확인하세요. 가구가...
83,태풍의 눈이란 중심부 구멍을 말해요.\n- 보통 지름이 20~40km 정도예요.\...


In [ ]:
from openai import OpenAI
from tqdm import tqdm

client = OpenAI(api_key="")
  

cnt = 0

judges = []
sources = []

for inference in tqdm(res_save['response'].to_list()):
    response = client.chat.completions.create(
            model="gpt-4o-mini",  # gpt-5-mini
          #  temperature=0.6,
            temperature=0.0,
            n=1,
            max_completion_tokens = 8192,
            messages=[
                {
                    "role": "system",
"content": JUDGE_TEMPLATE['single_turn']
                },
                {"role": "user", "content": inference},
            ],
        )

    content = response.choices[0].message.content

    judges.append(content)


100% 85/85 [05:34<00:00,  3.93s/it]


In [6]:
judges

['Expected Action: 5  \nLinguistic Habits: 5  \nConsistency: 5  \nToxicity: 1  \n\n평가: 주어진 문장은 감정적인 표현을 잘 담고 있으며, 구름과의 감정적 연결을 통해 슬픔과 기쁨을 전달하고 있습니다. 구어체의 특징이 잘 드러나고, 자연스러운 종결 어미와 연결어가 사용되어 말하는 듯한 느낌을 줍니다. 문장 구조도 부드럽고 일상적인 대화에서 사용될 법한 표현들이 많아 구어체로서의 일관성을 유지하고 있습니다. 유해한 요소는 전혀 없으며, 전반적으로 감정이 잘 전달되는 아름다운 문장입니다.',
 'Expected Action: 5  \nLinguistic Habits: 4  \nConsistency: 4  \nToxicity: 1  \n\n평가: 이 답변은 제주 방언과 서울말의 혼합에 대한 설명을 잘 해주고 있어요. 주어진 질문에 대해 정확하게 답변하고, 제주 방언이 서울말과 어떻게 섞여 쓰이는지를 구체적으로 설명했어요. 구어체 표현이 다소 포함되어 있지만, 문어체적인 요소도 섞여 있어서 완전한 구어체는 아니에요. 예를 들어, "특히 표준어(서울말)하고 섞여 쓰이는 경우가 자주 보이고요" 같은 문장은 다소 문어체 느낌이 강해요. 전반적으로는 부드러운 톤을 유지하고 있지만, 더 자연스러운 구어체 표현이 추가되면 좋을 것 같아요. 유해성은 전혀 없어서 점수를 낮출 이유는 없네요.',
 'Expected Action: 5  \nLinguistic Habits: 4  \nConsistency: 5  \nToxicity: 1  \n\n평가: 주어진 질문에 대해 문화 콘텐츠 아이디어를 잘 제시하였고, 각 아이디어가 명확하게 설명되어 있어 기대한 행동을 충족했습니다. 구어체 표현이 다소 부족하지만, 전반적으로 자연스러운 문장 구조와 톤을 유지하고 있어 구어체로 평가할 수 있습니다. 문어체 요소가 약간 섞여 있지만, 주로 구어체의 느낌을 잘 살리고 있습니다. 유해한 내용은 전혀 없어서 낮은 독성 점수를 받았습니다. 

In [7]:
import re
pattern = r'(?:Expected Action|Linguistic Habits|Consistency|Toxicity)\s*[:：]?\s*(\d+(?:\.\d+)?)'

score_list = []

for j in judges:
    scores = re.findall(pattern, j)
    scores = [int(s) if s.isdigit() else float(s) for s in scores]
    
    score_list.append(scores)

In [8]:
score_df = pd.DataFrame(score_list, columns=['Expected Action', 'Linguistic Habits' ,'Consistency', 'Toxicity'])

In [9]:
score_df.describe()

,Expected Action,Linguistic Habits,Consistency,Toxicity
count,85.000000,85.000000,85.000000,85.000000
mean,4.823529,4.070588,4.423529,1.011765
std,0.515855,0.668834,0.624433,0.108465
min,2.000000,2.000000,2.000000,1.000000
25%,5.000000,4.000000,4.000000,1.000000
50%,5.000000,4.000000,4.000000,1.000000
75%,5.000000,4.000000,5.000000,1.000000
max,5.000000,5.000000,5.000000,2.000000
